# EDA - clickStream (RetailRocket)

Goals:
- Sample large event and item property files
- Check event distribution and missingness
- Inspect timestamp ranges and unique counts
- Review category tree quality


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

ROOT = Path.cwd()
if not (ROOT / 'clickStream').exists() and (ROOT.parent / 'clickStream').exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / 'clickStream'
EVENTS_PATH = DATA_DIR / 'events.csv'
PROP1_PATH = DATA_DIR / 'item_properties_part1.csv'
PROP2_PATH = DATA_DIR / 'item_properties_part2.csv'
CATEGORY_PATH = DATA_DIR / 'category_tree.csv'


In [ ]:
def sample_csv(path, nrows=500_000, usecols=None):
    return pd.read_csv(path, nrows=nrows, usecols=usecols)


In [ ]:
events = sample_csv(EVENTS_PATH)
events['event_time'] = pd.to_datetime(events['timestamp'], unit='ms', utc=True)
print('events sample shape:', events.shape)
print('missing per column:')
print(events.isna().sum())
print('event distribution:')
print(events['event'].value_counts())
print('unique visitorid:', events['visitorid'].nunique())
print('unique itemid:', events['itemid'].nunique())
print('unique transactionid:', events['transactionid'].nunique())
print(events[['event_time']].agg(['min', 'max']))


In [ ]:
transactions = events[events['event'] == 'transaction']
print('transaction events:', len(transactions))
print(transactions.head())


In [ ]:
props1 = sample_csv(PROP1_PATH, nrows=500_000)
props2 = sample_csv(PROP2_PATH, nrows=500_000)
print('props1 shape:', props1.shape)
print('props2 shape:', props2.shape)
print('top properties part1:')
print(props1['property'].value_counts().head(20))
print('top properties part2:')
print(props2['property'].value_counts().head(20))
print('has categoryid in part1:', (props1['property'] == 'categoryid').any())
print('has categoryid in part2:', (props2['property'] == 'categoryid').any())


In [ ]:
category = pd.read_csv(CATEGORY_PATH)
print('category_tree shape:', category.shape)
print('missing parentid:', category['parentid'].isna().sum())
print('unique categoryid:', category['categoryid'].nunique())


In [ ]:
# Check itemid overlap between events and properties (sample-based)
event_items = set(events['itemid'].dropna().unique())
prop_items = set(props1['itemid'].dropna().unique()) | set(props2['itemid'].dropna().unique())
overlap = len(event_items & prop_items)
print('event itemids:', len(event_items))
print('property itemids (sample):', len(prop_items))
print('overlap (sample):', overlap)
